# Test Story Comic Generator

Use this notebook to:

- inspect the parsed `test/plot` chapters
- configure style and model parameters
- generate one comic page per chapter
- display the final 8-page comic inline


In [1]:
from IPython.display import HTML, Markdown, display

from test_story_notebook_utils import (
    DEFAULT_API_BASE_URL,
    DEFAULT_CHARACTER_ALIASES,
    build_generated_comic_html,
    build_story_overview_html,
    dump_json,
    generate_test_story_project,
    get_json,
    load_test_story_assets,
)


In [2]:
API_BASE_URL = DEFAULT_API_BASE_URL
PROJECT_TITLE = "图书馆禁区"
PROJECT_DESCRIPTION = ""
STYLE_ID = "noir"  # 画风；也可以换 american-modern / noir / vintage
IMAGE_PROFILE_ID = "gemini-image"
ALLOW_FALLBACK = False  # 建议关掉，避免失败时悄悄退回 mock 图
VERBOSE_LOGGING = True
NEGATIVE_PROMPT = "blurry unreadable text duplicated faces bad hands watermark"

RUNTIME_IMAGE_MODEL_CONFIG = {
    "model": "gemini-3-pro-image-preview",
    "imageSize": "1K",
    "aspectRatio": "3:4",
}

CHARACTER_ALIASES = dict(DEFAULT_CHARACTER_ALIASES)
SCENE_ALIASES = {}

CHAPTER_TEXT_OVERRIDES = {
    1: "",  # 如果 chapter1.txt 还是空的，这里要补内容
}


In [3]:
story_assets = load_test_story_assets(chapter_text_overrides=CHAPTER_TEXT_OVERRIDES)
display(Markdown("## Parsed Chapters"))
display(HTML(build_story_overview_html(story_assets)))

try:
    presets = get_json("/api/comics/presets", API_BASE_URL)
    display(Markdown("## Available Style Presets"))
    display(presets)
except Exception as exc:
    print(f"Could not fetch API presets from {API_BASE_URL}: {exc}")
    print("Start the local workflow server before running the generation cells.")


## Parsed Chapters

Chapter,Title,Location,Characters,Location Shift
1,查无此人,学校-图书馆,"伏黑惠, 钉崎野蔷薇",
2,钉崎的启示,图书馆,"伏黑惠, 钉崎野蔷薇",
3,不存在的禁区,图书馆,"伏黑惠, 钉崎野蔷薇",
4,谜之声的指引,图书馆,"伏黑惠, 钉崎野蔷薇",
5,不可阅读之书,图书馆,"伏黑惠, 钉崎野蔷薇",
6,禁区,图书馆,"伏黑惠, 钉崎野蔷薇",
7,禁之卷宗,图书馆,"伏黑惠, 钉崎野蔷薇, 夜蛾正道",
8,禁区的守⻔人,图书馆,"伏黑惠, 钉崎野蔷薇, 夜蛾正道",操场


## Available Style Presets

{'styles': [{'id': 'american-modern',
   'name': 'American Modern',
   'prompt': 'contemporary American superhero comic style, bold vibrant colors, dynamic heroic poses, detailed muscular anatomy, cinematic action scenes, modern digital art'},
  {'id': 'manga',
   'name': 'Manga',
   'prompt': 'Japanese manga style, clean precise black linework, screen tone shading, expressive eyes, dynamic speed lines, black and white with impact effects'},
  {'id': 'noir',
   'name': 'Noir',
   'prompt': 'film noir style, high contrast black and white, deep dramatic shadows, 1940s detective aesthetic, heavy bold inking, moody atmospheric lighting'},
  {'id': 'vintage',
   'name': 'Vintage',
   'prompt': 'Golden Age retro comic style, visible halftone dots, limited nostalgic palette, warm paper texture, classic adventure comic feeling'},
  {'id': 'color_manga',
   'name': 'Color Manga',
   'prompt': 'full-color Japanese manga style, clean precise linework, vibrant cel shading, expressive eyes, dynamic

In [4]:
display(Markdown("## Style Validation"))

if "presets" not in globals():
    presets = get_json("/api/comics/presets", API_BASE_URL)

available_style_ids = [item["id"] for item in presets.get("styles", [])]
print("Requested STYLE_ID:", STYLE_ID)
print("Available style ids:", available_style_ids)

if STYLE_ID not in available_style_ids:
    raise ValueError(
        f"STYLE_ID '{STYLE_ID}' was not found in /api/comics/presets. "
        "If you recently edited style_presets.json, restart the server and rerun this cell."
    )

resolved_style = next(item for item in presets["styles"] if item["id"] == STYLE_ID)
print("Resolved style:")
print(dump_json(resolved_style))


## Style Validation

Requested STYLE_ID: noir
Available style ids: ['american-modern', 'manga', 'noir', 'vintage', 'color_manga', 'chibi_color_manga', 'pop_art']
Resolved style:
{
  "id": "noir",
  "name": "Noir",
  "prompt": "film noir style, high contrast black and white, deep dramatic shadows, 1940s detective aesthetic, heavy bold inking, moody atmospheric lighting"
}


In [ ]:
result = generate_test_story_project(
    story_assets=story_assets,
    api_base_url=API_BASE_URL,
    project_title=PROJECT_TITLE,
    project_description=PROJECT_DESCRIPTION or None,
    style_id=STYLE_ID,
    image_profile_id=IMAGE_PROFILE_ID,
    runtime_image_model_config=RUNTIME_IMAGE_MODEL_CONFIG,
    negative_prompt=NEGATIVE_PROMPT,
    allow_fallback=ALLOW_FALLBACK,
    verbose=VERBOSE_LOGGING,
    character_aliases=CHARACTER_ALIASES,
    scene_aliases=SCENE_ALIASES,
)

display(Markdown(f"## Generated Comic: `{result['comicId']}`"))
print(
    dump_json(
        {
            "comicId": result["comicId"],
            "title": result["project"]["title"],
            "pageCount": result["project"]["pageCount"],
            "style": result["project"]["style"],
        }
    )
)


[1/8] Generating page 1 from chapter 1: 查无此人


In [ ]:
display(Markdown("## Rendered Pages"))
display(HTML(build_generated_comic_html(result["project"], api_base_url=API_BASE_URL, max_width=520)))
